# Update the electricity supply mix

This workflow shows how to overwrite the electricity generation mix of an IOT
*database* with up-to-date shares, either derived automatically from
[EMBER](https://ember-energy.org/) electricity-generation data or provided
explicitly.

It applies to any IOT database that exposes a **disaggregated power sector**,
i.e. separate generation technologies rather than a single aggregated
electricity sector. At the moment these are:

* **EXIOBASE** (IOT, monetary), used in the example below;
* **EMERGING-E**.

MARIO detects the disaggregated generation sectors automatically, so the same
call works across the supported databases without extra configuration.

Core methods
------------

The two main entry points are:

* `Database.update_supply_mix_iot` to rewrite the mix in place across the
  intersectoral (`z`) and final-demand (`Y`) blocks of one scenario;
* `Database.get_mix` to read back the current production mix of a sector
  bundle as a `region -> {sector: share}` mapping.

`Database.update_mix_iot` is kept as a backward-compatible alias of
`update_supply_mix_iot`.

> **Note:** Both methods are currently implemented for **IOT** tables only.

## What the update does

The EMBER taxonomy is more aggregated than the disaggregated electricity
sectors exposed by EXIOBASE IOT and EMERGING-E. When you request the
`"electricity"` mode, MARIO:

1. aggregates the database electricity bundle to the compatible EMBER fuel
   groups (coal, gas, nuclear, hydro, wind, solar, bioenergy, other
   renewables, other fossil);
2. reads the EMBER generation shares for the requested `year`;
3. redistributes each EMBER group total back to the original database
   sectors using the **current internal composition** of each group.

This means detailed technologies such as geothermal, tide/wave/ocean and
solar thermal are updated only through their parent EMBER group, while
transmission and distribution sectors are left unchanged.

## EMBER-based electricity mix

Parse an EXIOBASE monetary IOT table (which ships the disaggregated
electricity generation sectors) and update the mix for one scenario. When
`year` is omitted, MARIO uses the latest year available in the packaged EMBER
snapshot.

In [ ]:
import mario

db = mario.parse_exiobase(
    table="IOT",
    unit="Monetary",
    path="/path/to/IOT_2022_ixi.zip",
    name="EXIOBASE 3",
    year=2022,
)

# Rewrite the electricity generation mix of the baseline scenario
db.update_supply_mix_iot("electricity", scenario="baseline")

Inspect the resulting mix for one region. `get_mix` accepts the special
string `"electricity"`, which resolves the same generation-sector bundle
used by the update. EXIOBASE regions are two-letter codes, e.g. `"IT"` for
Italy.

In [ ]:
mix = db.get_mix("electricity", scenario="baseline")
mix["IT"]

### Choosing the reference year

Pass `year=...` to target a specific EMBER year. When a covered region has no
observation for that year, MARIO falls back to the nearest available year for
that region and logs the substitution.

In [ ]:
db.update_supply_mix_iot("electricity", scenario="baseline", year=2023)

### Creating the scenario on the fly

To write the updated mix into a new scenario, pass `clone_from` with an
existing source scenario. MARIO clones it first and then applies the update.

In [ ]:
db.update_supply_mix_iot(
    "electricity",
    scenario="ember_2025",
    clone_from="baseline",
)

## Aggregated regions

Aggregate regions are resolved to their underlying countries before the EMBER
generation is summed, so no manual bookkeeping is required. This covers two
situations:

* **Built-in Rest-of-World regions.** EXIOBASE ships aggregate regions such as
  `WA` (Rest of the World Asia and Pacific); MARIO expands them to their
  packaged member countries automatically.
* **User aggregations.** When you cluster countries yourself with `aggregate`
  (for example into a `Rest of Europe` region), MARIO reuses the region
  aggregation map stored on `db.meta.region_aggregation_map`.

If the map is not available (for example when the regions were renamed outside
of `aggregate`), pass the concordance explicitly through `region_aggregation`.
It accepts the same inputs as `aggregate`: a Region aggregation workbook, a
pandas `Series`/`DataFrame`, or a mapping.

In [ ]:
db.update_supply_mix_iot(
    "electricity",
    scenario="baseline",
    region_aggregation={
        "Rest of Europe": ["FR", "DE", "ES", "IT"],
    },
)

> **Note:** Regions that cannot be matched to the EMBER concordance, that are
> missing from the EMBER snapshot, or that report no positive generation are
> left unchanged. MARIO reports each of these cases in the logs.

## Custom shares

Instead of `"electricity"`, you can pass an explicit mapping
`region -> {sector: share}`. The combined coefficient rows of the selected
sectors are redistributed within each region according to the provided
weights. Shares are expected to sum to one; pass `rescale=True` to normalise
arbitrary positive totals.

The sector labels must match those of the database. You can list the available
electricity generation sectors from the mix keys:

In [ ]:
list(db.get_mix("electricity")["IT"])

In [ ]:
shares = {
    "IT": {
        "Production of electricity by solar photovoltaic": 0.6,
        "Production of electricity by wind": 0.4,
    },
}

db.update_supply_mix_iot(shares, scenario="baseline", rescale=True)

## Exporting to EMBER groups

Set `aggregate_as_ember=True` (only with `"electricity"`) to collapse the
disaggregated electricity sectors into the compatible EMBER groups right
after updating the mix, yielding a database whose electricity taxonomy
matches EMBER.

In [ ]:
db.update_supply_mix_iot(
    "electricity",
    scenario="baseline",
    aggregate_as_ember=True,
)